# Cuaderno U1-01. Concepto e importancia de la modelacion y la simulacion

**Modelacion y Simulacion Computacional** · Maestria en Ingenieria · Universidad de Sucre

Unidad 1, Fundamentos de modelacion en ingenieria · Subtema 1.1 del plan de asignatura

Docente Daniel David Otero Meza · Periodo 2026-2

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad1/U1_01_concepto_e_importancia.ipynb)


Este cuaderno acompana la Seccion 1.1 del libro. Separa el acto de modelar del acto de simular, muestra que la pregunta de ingenieria precede al modelo y reproduce con codigo la decision de la estacion de bombeo con la que el libro justifica el costo de modelar. Cada cifra que aqui se calcula se contrasta contra la que el libro publica.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante estara en capacidad de hacer lo siguiente.

1. Distinguir modelar de simular y explicar por que la facilidad de la simulacion no es evidencia de la calidad del modelo.
2. Traducir un enunciado vago en una pregunta de ingenieria con variable de interes, horizonte, resolucion y tolerancia.
3. Reproducir el calculo de la estacion de bombeo de la Seccion 1.1 y verificar sus cifras contra las publicadas en el libro.
4. Cuantificar el beneficio de una decision de diseno con un valor presente neto y decidir si la eleccion resiste la incertidumbre de un dato economico.

## Puesta a punto

La primera celda detecta el entorno e instala unicamente lo que falte. La segunda fija la semilla del curso y la paleta del libro. La tercera define las funciones de verificacion que se usan mas abajo. Ejecutelas en orden antes de continuar.

In [ ]:
# Puesta a punto del entorno. Detecta Colab e instala solo lo que falte.
import importlib
import importlib.util
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes):
    """Instala los paquetes ausentes sin reinstalar los que ya estan."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)
    return faltantes


AUSENTES = asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
                     "matplotlib": "matplotlib", "sympy": "sympy"})

print("Entorno de ejecucion:", "Google Colab" if EN_COLAB else "JupyterLab local")
print("Paquetes instalados en esta sesion:", AUSENTES or "ninguno, ya estaban")

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

# Semilla unica de la asignatura. Ningun resultado depende de una corrida.
SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

# Paleta del libro. Los cuadernos usan los mismos colores que las figuras.
PALETA = {
    "azul": "#1F4E79",
    "rojo": "#B3251E",
    "verde": "#2E7D32",
    "naranja": "#E07B00",
    "gris": "#5A5A5A",
    "morado": "#6A3D9A",
}

plt.rcParams.update({
    "figure.figsize": (8.6, 4.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
    "font.size": 10.0,
    "legend.frameon": True,
    "legend.framealpha": 0.92,
})

print(f"NumPy {np.__version__} · SciPy {scipy.__version__} · pandas {pd.__version__}")
print(f"SymPy {sp.__version__} · semilla del curso {SEMILLA}")

In [ ]:
# Bandera de revision de los ejercicios guiados.
# Mientras valga False el cuaderno se ejecuta completo aunque falten celdas.
# Pongala en True cuando haya completado las celdas marcadas para completar.
REVISAR = False
print("REVISAR =", REVISAR)

In [ ]:
def verificar_libro(nombre, obtenido, publicado, tolerancia=1e-3, unidad=""):
    """Contrasta un resultado calculado con la cifra que publica el libro."""
    valor = float(obtenido)
    escala = abs(publicado) if publicado else 1.0
    error = abs(valor - publicado) / escala
    print(f"{nombre:<46s} calculado {valor:>12.6g} {unidad:<10s}"
          f" libro {publicado:>12.6g}  error rel. {error:.1e}")
    assert error <= tolerancia, f"{nombre} se aparta de la cifra publicada"
    return valor


def comprobar(nombre, obtenido, referencia, tolerancia=1e-3, unidad=""):
    """Revisa una celda de ejercicio contra su valor de referencia.

    Con REVISAR en False solo informa que el ejercicio sigue pendiente, de modo
    que el cuaderno nunca se detiene por una celda sin completar.
    """
    if not REVISAR:
        print(f"[pendiente]  {nombre}")
        return False
    valor = float(obtenido)
    escala = abs(referencia) if referencia else 1.0
    error = abs(valor - referencia) / escala
    marca = "correcto " if error <= tolerancia else "revisar  "
    print(f"[{marca}]  {nombre} = {valor:.6g} {unidad}"
          f"  referencia {referencia:.6g}  error rel. {error:.2e}")
    assert error <= tolerancia, f"{nombre} no coincide con la referencia"
    return True


def ruta_datos(nombre):
    """Ubica un archivo de la carpeta datos sin usar rutas absolutas.

    Funciona igual en Colab, donde el cuaderno suele abrirse en el directorio
    de trabajo, y en una copia local del repositorio, donde el cuaderno vive
    dentro de Unidad1 o de soluciones.
    """
    candidatas = (Path("datos"),
                  Path("..") / "datos",
                  Path("..") / ".." / "datos",
                  Path("03_cuadernos") / "datos")
    for base in candidatas:
        if (base / nombre).exists():
            return base / nombre
    for base in candidatas:        # la carpeta existe pero el archivo aun no
        if base.is_dir():
            return base / nombre
    return Path("datos") / nombre  # entorno nuevo, como una sesion de Colab


def cargar_o_generar(nombre, generador):
    """Lee el archivo de datos y, si no esta, lo reconstruye con la semilla."""
    ruta = ruta_datos(nombre)
    if ruta.exists():
        print(f"Datos leidos de {ruta}")
        return pd.read_csv(ruta)
    tabla = generador()
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(ruta, index=False)
    print(f"Datos regenerados con la semilla {SEMILLA} y guardados en {ruta}")
    return tabla


def integrar_trapecio(valores, muestras):
    """Regla del trapecio compatible con NumPy 1 y con NumPy 2."""
    regla = getattr(np, "trapezoid", None) or np.trapz
    return float(regla(valores, muestras))


print("Funciones auxiliares disponibles.")

## 1. Modelar y simular no son el mismo acto

La Definicion 1.1 del libro dice que un modelo es una representacion
simplificada de un sistema real, formulada con un proposito declarado, cuya
validez se juzga siempre frente a ese proposito y nunca en abstracto. La
Definicion 1.2 dice que una simulacion es la ejecucion de ese modelo bajo un
conjunto especifico de entradas, parametros y condiciones auxiliares.

La consecuencia practica es incomoda. Un modelo mal construido se simula sin
dificultad y produce numeros perfectamente presentables. La celda siguiente lo
exhibe con dos modelos del mismo sistema, un tanque que se llena mientras pierde
caudal por una fuga proporcional al nivel. El primero conserva la masa y el
segundo olvida la fuga. Los dos se simulan igual de bien y solo uno cierra el
balance.

In [ ]:
from scipy.integrate import solve_ivp

AREA_TANQUE = 2.0        # m2
CAUDAL_ENTRADA = 0.004   # m3/s
COEFICIENTE_FUGA = 0.002  # m2/s, fuga proporcional al nivel


def modelo_con_fuga(t, x):
    """Balance de volumen con la fuga incluida. Estado, nivel en m."""
    nivel, = x
    return [(CAUDAL_ENTRADA - COEFICIENTE_FUGA * nivel) / AREA_TANQUE]


def modelo_sin_fuga(t, x):
    """El mismo tanque con un proceso omitido. Sigue siendo simulable."""
    return [CAUDAL_ENTRADA / AREA_TANQUE]


def simular(modelo, horizonte=3600.0, nivel_inicial=0.0):
    """Ejecuta el modelo bajo un escenario concreto."""
    return solve_ivp(modelo, (0.0, horizonte), [nivel_inicial],
                     dense_output=True, rtol=1e-9, atol=1e-12)


tiempo = np.linspace(0.0, 3600.0, 400)
con_fuga = simular(modelo_con_fuga).sol(tiempo)[0]
sin_fuga = simular(modelo_sin_fuga).sol(tiempo)[0]

# Verificacion, el balance de volumen debe cerrar en el modelo completo. El
# residuo que queda es el error de la regla del trapecio con la que se integra
# la fuga, y no un defecto del modelo, de modo que se mide en relacion con el
# volumen que entra y se comprueba que baja al refinar el muestreo.
def cierre_del_balance(puntos):
    """Residuo relativo del balance de volumen, adimensional."""
    malla = np.linspace(0.0, 3600.0, puntos)
    nivel = simular(modelo_con_fuga).sol(malla)[0]
    entrada = CAUDAL_ENTRADA * 3600.0
    fuga = integrar_trapecio(COEFICIENTE_FUGA * nivel, malla)
    return abs(entrada - fuga - AREA_TANQUE * nivel[-1]) / entrada


volumen_final = AREA_TANQUE * con_fuga[-1]
entrada_total = CAUDAL_ENTRADA * 3600.0
fuga_total = integrar_trapecio(COEFICIENTE_FUGA * con_fuga, tiempo)
cierre = cierre_del_balance(400)

figura, eje = plt.subplots()
eje.plot(tiempo / 60, con_fuga, color=PALETA["azul"], lw=1.9,
         label="Modelo que conserva la masa")
eje.plot(tiempo / 60, sin_fuga, color=PALETA["rojo"], lw=1.7, ls="--",
         label="Modelo con la fuga omitida")
eje.set_xlabel("Tiempo transcurrido (min)")
eje.set_ylabel("Nivel en el tanque (m)")
eje.legend(loc="upper left")
eje.set_title("Dos modelos del mismo tanque, simulados con igual facilidad")
plt.show()

print(f"Nivel a los 60 min, modelo completo   {con_fuga[-1]:8.4f} m")
print(f"Nivel a los 60 min, modelo incompleto {sin_fuga[-1]:8.4f} m")
print(f"Diferencia relativa                   {abs(sin_fuga[-1] - con_fuga[-1]) / con_fuga[-1]:8.1%}")
print(f"Entra {entrada_total:.3f} m3, se fuga {fuga_total:.3f} m3 y quedan "
      f"{volumen_final:.3f} m3 en el tanque")
for puntos in (400, 1600, 6400):
    print(f"  con {puntos:>5d} puntos de muestreo el residuo relativo del "
          f"balance vale {cierre_del_balance(puntos):.2e}")
assert cierre < 1e-4, "el modelo que conserva masa debe cerrar el balance"
print("\nEl residuo cae al refinar el muestreo, de modo que es error numerico y "
      "no un proceso que falte en el modelo.")

La curva roja es suave, monotona y de aspecto respetable. Nada en ella delata
que el modelo que la produjo olvida un proceso. Lo que la delata es el balance,
que en el modelo completo cierra en el orden del error de integracion y en el
incompleto no cierra en absoluto, porque el volumen que entra no aparece por
ningun lado.

## 2. La pregunta de ingenieria precede al modelo

El libro insiste en que la primera obligacion del modelador no es escribir
ecuaciones sino escribir la pregunta, con su variable de interes, su horizonte
temporal, su resolucion espacial y su tolerancia admisible. El Problema 1-3 del
capitulo pide justamente eso para un cuarto frio. La celda siguiente organiza el
contraste entre un enunciado vago y una pregunta de ingenieria.

In [ ]:
PREGUNTAS = [
    {"componente": "variable de interes",
     "enunciado vago": "el comportamiento termico",
     "pregunta de ingenieria": "temperatura del aire en el punto mas caliente de la camara"},
    {"componente": "horizonte",
     "enunciado vago": "no se declara",
     "pregunta de ingenieria": "las 6 h siguientes a un corte de energia"},
    {"componente": "resolucion",
     "enunciado vago": "no se declara",
     "pregunta de ingenieria": "un valor cada 10 min, con un solo volumen de control"},
    {"componente": "tolerancia",
     "enunciado vago": "no se declara",
     "pregunta de ingenieria": "0.5 C sobre la temperatura maxima alcanzada"},
    {"componente": "decision que sostiene",
     "enunciado vago": "ninguna en concreto",
     "pregunta de ingenieria": "si conviene instalar una planta de respaldo"},
]
pd.DataFrame(PREGUNTAS).set_index("componente")

La columna de la derecha determina casi todo lo demas. Una tolerancia de
0.5 grados y un solo volumen de control descartan de entrada un modelo de
parametros distribuidos, y un horizonte de seis horas obliga a un modelo
dinamico. La pregunta, y no el gusto del modelador, fija el tipo de modelo.

## 3. El costo de modelar frente al costo de la decision

Aqui se reproduce el calculo de la Seccion 1.1 del libro. Una estacion de bombeo
eleva 45 L/s de agua potable por 1200 m de tuberia de PVC hasta un tanque
situado 28 m por encima del pozo de succion, y el proyectista duda entre un
diametro de 200 mm y uno de 250 mm.

El modelo consta de las tres relaciones de la Ecuacion 1.1 del libro, que son la
altura total de bombeo, la velocidad media y la energia anual. El factor de
friccion sale de la ecuacion implicita de Colebrook y White, que se resuelve con
la rutina de busqueda de raiz de SciPy.

In [ ]:
from scipy.optimize import brentq

GRAVEDAD = 9.81          # m/s2
DENSIDAD = 998.2         # kg/m3, agua a 20 C
VISCOSIDAD_CINEMATICA = 1.004e-6   # m2/s, agua a 20 C

CAUDAL = 0.045           # m3/s
LONGITUD = 1200.0        # m
ALTURA_ESTATICA = 28.0   # m
RUGOSIDAD = 1.5e-6       # m, PVC
RENDIMIENTO = 0.72       # adimensional, conjunto motobomba
HORAS_ANUALES = 18 * 365  # h/ano
TARIFA = 0.20            # USD/kWh
SOBRECOSTO_250 = 21000.0  # USD
TASA = 0.08              # adimensional
VIDA_UTIL = 25           # anos


def velocidad_media(caudal, diametro):
    """Velocidad media en la conduccion, en m/s."""
    return 4.0 * caudal / (np.pi * diametro**2)


def factor_friccion(reynolds, rugosidad, diametro):
    """Factor de friccion de Darcy por Colebrook y White, adimensional."""
    def residuo(f):
        return (1.0 / np.sqrt(f)
                + 2.0 * np.log10(rugosidad / (3.7 * diametro)
                                 + 2.51 / (reynolds * np.sqrt(f))))
    return brentq(residuo, 5e-3, 0.1, xtol=1e-14)


def altura_bombeo(diametro, caudal=CAUDAL):
    """Altura total de bombeo y perdida por friccion, ambas en m."""
    velocidad = velocidad_media(caudal, diametro)
    reynolds = velocidad * diametro / VISCOSIDAD_CINEMATICA
    friccion = factor_friccion(reynolds, RUGOSIDAD, diametro)
    perdida = friccion * (LONGITUD / diametro) * velocidad**2 / (2 * GRAVEDAD)
    return ALTURA_ESTATICA + perdida, perdida, velocidad, friccion, reynolds


def energia_anual(diametro, caudal=CAUDAL):
    """Energia anual que exige la impulsion, en kWh por ano."""
    altura = altura_bombeo(diametro, caudal)[0]
    potencia = DENSIDAD * GRAVEDAD * caudal * altura / RENDIMIENTO   # W
    return potencia * HORAS_ANUALES / 1000.0


filas = []
for diametro in (0.200, 0.250):
    altura, perdida, velocidad, friccion, reynolds = altura_bombeo(diametro)
    filas.append({"diametro (mm)": diametro * 1000,
                  "velocidad (m/s)": round(velocidad, 4),
                  "Reynolds": round(reynolds, 0),
                  "factor de friccion": round(friccion, 5),
                  "perdida (m)": round(perdida, 4),
                  "altura total (m)": round(altura, 4),
                  "energia (kWh/ano)": round(energia_anual(diametro), 1)})
comparacion = pd.DataFrame(filas).set_index("diametro (mm)")
comparacion

### Verificacion contra las cifras del libro

El libro reporta, para 200 mm, una velocidad de 1.432 m/s, una perdida por
friccion de 9.21 m y un consumo de 149619 kWh anuales, y para 250 mm una
velocidad de 0.917 m/s, una perdida de 3.14 m y un consumo de 125231 kWh. La
diferencia es de 24388 kWh al ano, que con una tarifa de 0.20 USD/kWh equivalen
a 4878 USD al ano.

In [ ]:
_, perdida_200, velocidad_200, _, _ = altura_bombeo(0.200)
_, perdida_250, velocidad_250, _, _ = altura_bombeo(0.250)
energia_200 = energia_anual(0.200)
energia_250 = energia_anual(0.250)
ahorro_energia = energia_200 - energia_250
ahorro_dinero = TARIFA * ahorro_energia

verificar_libro("velocidad con 200 mm", velocidad_200, 1.432, 1e-3, "m/s")
verificar_libro("perdida por friccion con 200 mm", perdida_200, 9.21, 1e-3, "m")
verificar_libro("energia anual con 200 mm", energia_200, 149619, 1e-3, "kWh")
verificar_libro("velocidad con 250 mm", velocidad_250, 0.917, 1e-3, "m/s")
verificar_libro("perdida por friccion con 250 mm", perdida_250, 3.14, 2e-3, "m")
verificar_libro("energia anual con 250 mm", energia_250, 125231, 1e-3, "kWh")
verificar_libro("ahorro anual de energia", ahorro_energia, 24388, 1e-3, "kWh")
verificar_libro("ahorro anual de dinero", ahorro_dinero, 4878, 1e-3, "USD")

### La verificacion por escalas

El libro no se conforma con el numero, sino que lo somete a una prueba
independiente. La perdida por friccion escala como el diametro elevado a menos
cinco, de modo que aumentar el diametro en un factor 1.25 deberia reducirla
cerca de 3.05 veces. La relacion calculada es 2.93, algo menor porque el factor
de friccion tambien cambia con el numero de Reynolds. Esa clase de comprobacion
de orden de magnitud es la primera defensa contra un error de programacion.

In [ ]:
razon_calculada = perdida_200 / perdida_250
razon_teorica = 1.25**5
verificar_libro("razon de perdidas entre 200 y 250 mm", razon_calculada, 2.93, 1e-3)
verificar_libro("razon que predice la escala D^-5", razon_teorica, 3.05, 1e-3)
print(f"\nLa escala pura sobrestima la reduccion en un "
      f"{100 * (razon_teorica / razon_calculada - 1):.1f} por ciento, "
      f"que es el efecto del factor de friccion.")

### La decision economica

Si el sobrecosto de la tuberia mayor asciende a 21000 USD, el libro reporta que
se recupera en 4.3 anos y produce un beneficio neto de 31068 USD descontado a
veinticinco anos con una tasa del 8 por ciento.

In [ ]:
def factor_anualidad(tasa, anios):
    """Valor presente de una anualidad unitaria, adimensional."""
    return (1.0 - (1.0 + tasa) ** (-anios)) / tasa


periodo_retorno = SOBRECOSTO_250 / ahorro_dinero
beneficio_neto = ahorro_dinero * factor_anualidad(TASA, VIDA_UTIL) - SOBRECOSTO_250

verificar_libro("periodo de retorno", periodo_retorno, 4.3, 5e-3, "anos")
verificar_libro("beneficio neto a 25 anos", beneficio_neto, 31068, 1e-3, "USD")
print(f"\nFactor de anualidad a {VIDA_UTIL} anos y {TASA:.0%} · "
      f"{factor_anualidad(TASA, VIDA_UTIL):.6f}")
print("El calculo cabe en media hora de trabajo y sostiene una decision de "
      f"{beneficio_neto:,.0f} USD.")

### La alternativa que no existe en ningun registro

Ninguna serie historica de la estacion habria podido responder la pregunta,
porque la alternativa de 250 mm no aparece en ningun dato. Ese es el aporte
propio de la simulacion, que consiste en recorrer un espacio de alternativas de
diseno antes de comprometer capital. La figura barre el diametro de forma
continua y situa sobre ella los dos candidatos.

In [ ]:
diametros = np.linspace(0.150, 0.400, 120)
energias = np.array([energia_anual(d) for d in diametros])
beneficios = np.array([
    TARIFA * (energia_200 - e) * factor_anualidad(TASA, VIDA_UTIL) for e in energias])

figura, (eje1, eje2) = plt.subplots(2, 1, figsize=(8.6, 6.4), sharex=True)
eje1.plot(diametros * 1000, energias / 1000, color=PALETA["azul"], lw=1.9)
eje1.plot([200, 250], [energia_200 / 1000, energia_250 / 1000], "o",
          color=PALETA["rojo"], ms=6.5, zorder=5)
eje1.annotate("200 mm", (200, energia_200 / 1000), textcoords="offset points",
              xytext=(10, 6), color=PALETA["rojo"])
eje1.annotate("250 mm", (250, energia_250 / 1000), textcoords="offset points",
              xytext=(10, 10), color=PALETA["rojo"])
eje1.set_ylabel("Energia anual (MWh/ano)")
eje1.set_title("Barrido continuo de una alternativa que no existe en los datos")

eje2.plot(diametros * 1000, beneficios / 1000, color=PALETA["verde"], lw=1.9,
          label="Valor presente del ahorro frente a 200 mm")
eje2.axhline(SOBRECOSTO_250 / 1000, color=PALETA["naranja"], lw=1.2, ls="--",
             label="Sobrecosto de la tuberia de 250 mm")
eje2.set_xlabel("Diametro interno de la impulsion (mm)")
eje2.set_ylabel("Miles de USD")
eje2.legend(loc="upper left")
plt.show()

print(f"Con 250 mm el ahorro descontado supera al sobrecosto en "
      f"{beneficio_neto:,.0f} USD.")

## 4. Ejercicios guiados

Las cinco celdas siguientes estan incompletas a proposito, cada una con la marca
`# COMPLETE:`, un valor de partida deliberadamente incorrecto y una celda de
verificacion inmediata. Complete las cinco, vuelva a la celda de la bandera,
ponga `REVISAR = True` y ejecute el cuaderno completo desde el principio.

### Ejercicio 1. Regimen de flujo en la impulsion

La Tabla 1.4 del libro fija el umbral del numero de Reynolds, con transicion
cerca de 2300 y turbulencia plena sobre 4000.

In [ ]:
# COMPLETE: escriba la funcion que devuelve el numero de Reynolds de la
# conduccion, definido como la velocidad media por el diametro sobre la
# viscosidad cinematica, y evaluela para el diametro de 200 mm.
def numero_reynolds(caudal, diametro):
    return float("nan")   # valor de partida deliberadamente incorrecto


reynolds_200 = numero_reynolds(CAUDAL, 0.200)

In [ ]:
comprobar("numero de Reynolds con 200 mm", reynolds_200, 285337.5, 1e-5)

### Ejercicio 2. El diametro que fija un criterio de velocidad

Muchos manuales de diseno limitan la velocidad en la impulsion a 1.0 m/s para
reducir el golpe de ariete. Ese criterio, por si solo, determina un diametro.

In [ ]:
# COMPLETE: despeje del caudal y de la velocidad el diametro interno, en metros,
# que produce exactamente una velocidad media de 1.0 m/s con el caudal de diseno.
VELOCIDAD_LIMITE = 1.0          # m/s
diametro_criterio = 0.0         # valor de partida deliberadamente incorrecto

In [ ]:
comprobar("diametro para 1.0 m/s", diametro_criterio, 0.2393654, 1e-5, "m")

### Ejercicio 3. El ahorro con una tarifa incierta

La tarifa de 0.20 USD/kWh es un dato de mercado y no una constante fisica. El
Problema 1-28 del capitulo pide repetir el analisis con una tarifa incierta en
un 20 por ciento, esto es entre 0.16 y 0.24 USD/kWh.

In [ ]:
# COMPLETE: calcule el ahorro anual en dolares de pasar de 200 mm a 250 mm para
# la tarifa baja y para la tarifa alta de la banda de incertidumbre.
TARIFA_BAJA, TARIFA_ALTA = 0.16, 0.24
ahorro_tarifa_baja = 0.0        # valor de partida deliberadamente incorrecto
ahorro_tarifa_alta = 0.0        # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("ahorro con tarifa baja", ahorro_tarifa_baja, 3902.10, 1e-4, "USD/ano")
dos = comprobar("ahorro con tarifa alta", ahorro_tarifa_alta, 5853.15, 1e-4, "USD/ano")

### Ejercicio 4. El valor presente neto como funcion reutilizable

In [ ]:
# COMPLETE: escriba la funcion que devuelve el valor presente neto de una
# alternativa, esto es el ahorro anual multiplicado por el factor de anualidad
# menos el sobrecosto de inversion, y aplicola al caso central de 250 mm.
def valor_presente_neto(ahorro_anual, sobrecosto, tasa=TASA, anios=VIDA_UTIL):
    return float("nan")   # valor de partida deliberadamente incorrecto


vpn_250 = valor_presente_neto(ahorro_dinero, SOBRECOSTO_250)

In [ ]:
comprobar("VPN de la alternativa de 250 mm", vpn_250, 31068, 1e-3, "USD")

### Ejercicio 5. Problema 1-28, un tercer diametro

El Problema 1-28 pide repetir el analisis con un tercer diametro y decidir si la
eleccion cambia. Tome 300 mm, cuyo sobrecosto respecto de la tuberia de 250 mm
asciende a 15000 USD, y evalue el valor presente neto en los tres puntos de la
banda de tarifa.

In [ ]:
# COMPLETE: calcule el ahorro anual de energia de pasar de 250 mm a 300 mm y,
# con la funcion del ejercicio anterior, el valor presente neto de esa mejora
# para la tarifa baja de 0.16 USD/kWh, con un sobrecosto de 15000 USD.
SOBRECOSTO_300 = 15000.0        # USD
ahorro_energia_300 = 0.0        # valor de partida deliberadamente incorrecto
vpn_300_baja = 0.0              # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("ahorro de energia de 300 frente a 250 mm",
                ahorro_energia_300, 7383.18, 1e-4, "kWh/ano")
dos = comprobar("VPN de 300 mm con tarifa baja", vpn_300_baja, -2389.79, 1e-3, "USD")
if REVISAR and uno and dos:
    print("\nConclusion, la segunda mejora no es robusta frente a la "
          "incertidumbre de la tarifa.")

## 5. Problemas del capitulo

### Problemas de apropiacion conceptual

Los Problemas 1-1 a 1-5 del capitulo son de redaccion y argumentacion, y se
responden en la celda de texto siguiente, no con codigo. Sirven de guia estas
indicaciones.

- **1-1.** Un modelo bien construido se simula de manera inadecuada cuando se
  ejecuta fuera de su dominio de validez o con un paso de tiempo que no resuelve
  la escala del fenomeno. Piense en el modelo de la camara de contacto del
  Ejemplo 1.1 integrado con un paso mayor que el tiempo de residencia.
- **1-2.** Reproducir con error inferior al 2 por ciento los datos con los que se
  calibro no es evidencia de validez, porque cualquier modelo con suficientes
  parametros libres reproduce cualquier conjunto de datos. La objecion exige
  datos independientes, segun la Seccion 1.3 del libro.
- **1-3.** Use la tabla de la seccion 2 de este cuaderno como plantilla.
- **1-4.** La creciente maxima de veinte anos describe la cuenca que existia,
  con su cobertura y su infraestructura. El dique cambia el sistema y con ello
  invalida parte de la evidencia, que es el segundo lazo de la Figura 1.2.
- **1-5.** La advertencia de Box invita a jerarquizar los errores, no a
  relativizarlos, y reportar la incertidumbre es precisamente la forma de decir
  cuanto pesa lo que el modelo esta equivocando.

Escriba sus respuestas en la celda siguiente, con una extension de un parrafo
por problema.

### Respuestas del estudiante a los Problemas 1-1 a 1-5

*Escriba aqui. Un parrafo por problema, con un ejemplo de su propia area en el
Problema 1-1 y una referencia explicita a la seccion del libro que sostiene cada
argumento.*

## Cierre

### Lista de comprobacion

Marque cada punto solo si puede hacerlo sin mirar el cuaderno.

- Explicar la diferencia entre modelar y simular con un ejemplo propio en el que un modelo incompleto produzca resultados de buen aspecto.
- Escribir una pregunta de ingenieria completa a partir de un enunciado vago.
- Reproducir el calculo de la estacion de bombeo y verificar sus cifras contra el libro.
- Comprobar un resultado numerico con un argumento de escala antes de aceptarlo.
- Decidir si una alternativa de diseno resiste la incertidumbre de un dato economico.

### Que revisar si algo no salio

- Si la verificacion contra el libro falla, revise primero las constantes, en especial la viscosidad cinematica del agua a 20 grados, que vale 1.004e-6 m2/s, y la densidad de 998.2 kg/m3.
- Si el factor de friccion no converge, compruebe que el intervalo de busqueda de `brentq` encierra la raiz y que el numero de Reynolds entra en la formula, no la velocidad.
- Si el valor presente neto no coincide, revise el factor de anualidad, que es la suma de los descuentos de los 25 anos y no el descuento del ano 25.
- Para la teoria, relea la Seccion 1.1 del libro, las Definiciones 1.1 y 1.2 y la Figura 1.2.

### Declaracion del uso de asistentes de programacion

Si empleo un asistente basado en modelos de lenguaje para resolver alguna celda, declarelo en la entrega, indique en cual y describa que prueba aplico para convencerse de que el codigo es correcto. La regla de la asignatura es que el estudiante responde por el resultado que firma, con independencia de quien escriba las lineas.